<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_RRAO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# FRTB Residual Risk Add-on (RRAO) Calculation
#
# This script uses pandas to replicate the RRAO calculation example
# based on Article 325u.
#
# The RRAO is a simple, non-netting capital charge applied to the
# gross notional amount of instruments with residual risks.

# 1. Import Pandas
import pandas as pd
import numpy as np

# 2. Define the Hypothetical Portfolio
# We create a portfolio based on the example, defining each instrument's
# notional and its RRAO risk category.
#
# * Exotic: Exotic Underlying (1.0% RW)
# * Other_Residual: Other Residual Risk (0.1% RW, non-exempt)
# * Exempt: Other Residual Risk (Exempt, e.g., exchange-listed)
# * Standard: Not subject to RRAO
data = [
    {'TradeID': 'Trade 1', 'Instrument': 'Weather Derivative (Swap)', 'Notional': 50000000, 'RiskCategory': 'Exotic'},
    {'TradeID': 'Trade 2', 'Instrument': 'OTC Barrier Option', 'Notional': 100000000, 'RiskCategory': 'Other_Residual'},
    {'TradeID': 'Trade 3', 'Instrument': 'Exchange-Listed Basket Option', 'Notional': 75000000, 'RiskCategory': 'Exempt'},
    {'TradeID': 'Trade 4', 'Instrument': 'Plain-Vanilla Call Option', 'Notional': 20000000, 'RiskCategory': 'Standard'}
]

# Create the DataFrame
df = pd.DataFrame(data)

print("--- Initial Portfolio ---")
print(df)


# 3. Define Risk Weights and Calculate Charge
# We map the risk categories to their corresponding risk weights as
# defined in Article 325u(3) and apply the exemption from Article 325u(4).
#
# * Exotic (Art. 325u(3)(a)): 1.0%
# * Other Residual (Art. 325u(3)(b)): 0.1%
# * Exempt/Standard: 0.0%

# Define the mapping of category to risk weight
risk_weight_map = {
    'Exotic': 0.01,
    'Other_Residual': 0.001,
    'Exempt': 0.0,
    'Standard': 0.0
}

# Create the 'RiskWeight' column using the map
df['RiskWeight'] = df['RiskCategory'].map(risk_weight_map)

# Calculate the 'CapitalCharge' for each instrument
df['CapitalCharge'] = df['Notional'] * df['RiskWeight']

print("\n--- Portfolio with Calculated Charges ---")

# Format for display (optional, but nice)
display_df = df.copy()
display_df['Notional'] = display_df['Notional'].apply(lambda x: f"£{x:,.0f}")
display_df['RiskWeight'] = display_df['RiskWeight'].apply(lambda x: f"{x:.1%}")
display_df['CapitalCharge'] = display_df['CapitalCharge'].apply(lambda x: f"£{x:,.0f}")

# Using to_string() for better alignment in console output
print(display_df.to_string(index=False))


# 4. Summarize Total Capital Charge
# The final RRAO charge is the simple sum of the individual capital charges.

# Calculate the total RRAO
total_rrao = df['CapitalCharge'].sum()

print(f"\nTotal RRAO Capital Charge: £{total_rrao:,.0f}")

# Show a summary by category
print("\n--- Summary by Category ---")
summary = df.groupby('RiskCategory')['CapitalCharge'].sum()

for category, charge in summary.items():
    if charge > 0:
        print(f"{category} Charge: £{charge:,.0f}")

--- Initial Portfolio ---
   TradeID                     Instrument   Notional    RiskCategory
0  Trade 1      Weather Derivative (Swap)   50000000          Exotic
1  Trade 2             OTC Barrier Option  100000000  Other_Residual
2  Trade 3  Exchange-Listed Basket Option   75000000          Exempt
3  Trade 4      Plain-Vanilla Call Option   20000000        Standard

--- Portfolio with Calculated Charges ---
TradeID                    Instrument     Notional   RiskCategory RiskWeight CapitalCharge
Trade 1     Weather Derivative (Swap)  £50,000,000         Exotic       1.0%      £500,000
Trade 2            OTC Barrier Option £100,000,000 Other_Residual       0.1%      £100,000
Trade 3 Exchange-Listed Basket Option  £75,000,000         Exempt       0.0%            £0
Trade 4     Plain-Vanilla Call Option  £20,000,000       Standard       0.0%            £0

Total RRAO Capital Charge: £600,000

--- Summary by Category ---
Exotic Charge: £500,000
Other_Residual Charge: £100,000
